# Oozie


Configuraciones clave de la máquina virtual

- 8GB de Memoria para la VM
- Configurar el Adaptador puente de Red para WIFI
- Portapapeles Bidireccional
- Tamaño monitor (150% o 200%)



### Preparar los ejemplos

1. Applications $\rightarrow$ File Browser $\rightarrow$ workspace
2. Find oozie-examples.tar.gz in File System 

>*Nota: (hay dos oozie-examples.tar.gz el almacenado en `* cdh *` y el almacenado en el despliegue en tomcat de la UI de oozie)*





3. Copy oozie-examples.tar.gz to ~\workspace
4. Extract Here ... oozie-examples.tar.gz
5. Go to examples/apps/distcp

### Ejemplo dist-cp

1. Web Browser $\rightarrow$ NameNode
2. Edit job.properties: 
    - NameNode -> `hdfs://quickstart.cloudera:8020`
    - jobtracker -> `quickstart.cloudera:8032`
3. View workflow.xml



## Ejecución


1. Colocar los ficheros necesarios (por extensión) en hdfs

```
cd workspace
hdfs dfs -put examples /user/cloudera
hdfs dfs -ls examples/*
```
>El objetivo es que la estructura de directorios que esta en job.properties y en workflow.xml esté en hdfs. 

También tienen que estar el fichero examples/input-data/text/data.txt que es nuestro fichero fuente, sino está dará error en la ejecución del proceso.

2. Ejecución


```
oozie job \
  -oozie http://localhost:11000/oozie \
  -config ~/workspace/examples/apps/distcp/job.properties \
  -run
```
2. Obtener información del proceso
```
oozie job 
  -oozie http://localhost:11000/oozie 
  -info <jobid>
```

3. Comprobar Oozie Web


4. Comprobar trabajo realizado

```
hdfs dfs -ls examples/output-data/distcp
```

5. Comprobar Hue $\rightarrow$ JobTracker



### Borrado



1. Borrar en local

```
cd ~/workspace
rm -rf examples
rm oo*
```

2. Borrar en HDFS



```
hdfs dfs -rm -r examples
```






# Oozie (Hue Editor)

0.1 Comprobamos con el MySQL de la máquina virtual la existencia de la Base de Datos.



```
mysql -u retail_dba;
show databases;
use retail_db;
show tables;
select * from orders limit 10;
```



## Preparación



1. En primer lugar hay que bajar el fichero pig de su ubicación en github y después de renombrarlo subirlo a hdfs. 



```
wget https://raw.githubusercontent.com/curso-iabd-uclm/hadoop/main/pig/proceso_orders.pig
hdfs dfs -put proceso_orders.pig

```

NOTA: si existe previamente el fichero.pig lo borramos `hdfs dfs -rm proceso_orders.pig`





In [ ]:
%%writefile proceso_orders.pig

orders = LOAD 'orders/part-m-00000' using PigStorage(';') AS (order_id:int, order_date: chararray, order_customer:int, order_status:chararray);
customer_orders  = GROUP orders  BY order_customer;
customer_orders_no = FOREACH customer_orders GENERATE group,COUNT($1) AS total;
customer_orders_no = ORDER customer_orders_no BY total DESC;

-- Almacenamos los resultados
STORE customer_orders_no INTO 'pig_out/' USING PigStorage (',');


Writing proceso_orders.pig


2. Aseguramos que los directorios de salida tanto del proceso sqoop como del proceso pig no existen previamente.

>Comprobamos si existe
`hdfs dfs -ls orders`

>Y si existe lo borramos

```
hdfs dfs -rm orders/* 
hdfs dfs -rmdir orders
```

>y lo mismo con el directorio de salida del proceso Pig

```
hdfs dfs -ls pig_out
hdfs dfs -rm pig_out/*
hdfs dfs -rmdir pig_out
```


3. Hue-Oozie no lee de la misma configuración y librerías que Oozie desde línea de comandos, eso hace que algunos procesos como **sqoop** no funcionen correctamente. Para evitar este problema hay que ejecutar los siguientes scripts. 

In [ ]:
%%bash
sudo -u hdfs hadoop fs -chown cloudera:cloudera /user/oozie/share/lib/lib_20171023091808/sqoop

hdfs dfs -put /var/lib/sqoop/mysql-connector-java.jar /user/oozie/share/lib/lib_20171023091808/sqoop

sudo -u hdfs hadoop fs -chown oozie:oozie /user/oozie/share/lib/lib_20171023091808/sqoop


oozie admin -oozie http://localhost:11000/oozie -sharelibupdate
oozie admin -oozie http://localhost:11000/oozie -shareliblist sqoop


5. Para comprobar el éxito del proceso se pueden ejecutar los siguientes scripts.

```
hdfs dfs -ls orders
hdfs dfs -ls pig_out
hdfs dfs -cat pig_out/*
```




## Ejecución

1. Comando Sqoop

sqoop import 
--connect jdbc:mysql://localhost/retail_db 
--username retail_dba 
--password cloudera 
--table orders -m 1


In [ ]:
sqoop import --connect jdbc:mysql://localhost/retail_db --username retail_dba --password cloudera --table orders -m 1


2. Script de Pig

In [ ]:
hdfs dfsadmin -report

In [ ]:
hdfs dfs -df -h